# CyanoHAB Dataset Builder (v7) — pure addition, new dates only

### What changed from the last version
**Nothing from your existing 2,154 real rows is touched, filtered, or reprocessed.** This version
only runs Earth Engine on brand-new dates and adds the result on top of your existing file. Zero
wasted work.

### Which new dates, and why
- **New bloom dates use the western-basin hotspot region** (same fix as before — the bloom zone is
  a small fraction of the full lake, so sampling just that area gets far more real bloom pixels per
  date than sampling the whole lake).
- Candidates below are drawn from NOAA's Lake Erie HAB severity history: **2022 (severity index
  6.8, peaked late August, persisted through mid-September, 416 sq mi at peak)** and **2023
  (severity 5.3, peaked mid-August, persisted to early September)** — both larger than the 2024
  season you already sampled. **Verify each date's actual cloud cover in the CyAN File Search /
  viewer before downloading** — a date with a severe bloom on the ground is only useful if
  Sentinel-2 actually got a clear look at it.

```
New bloom dates (western-basin hotspot):
  2022-08-25   2022-09-05   2022-09-10
  2023-08-15   2023-08-25

New calm dates (full lake, for additional no-bloom diversity):
  2022-05-15   2023-05-15
```

Adjust this list after checking the viewer — swap out any date with heavy cloud cover for a
nearby one within a few days.


In [ ]:
from google.colab import files
import os
import shutil

# Create the folder expected by the pipeline
CYAN_DIR = "/content/cyan_files"
os.makedirs(CYAN_DIR, exist_ok=True)

print("Select the 7 CyAN .tif files AND your existing CSV.")
uploaded = files.upload()

# Move TIFF files into cyan_files/
csv_files = []

for filename in uploaded.keys():
    if filename.lower().endswith(".tif") or filename.lower().endswith(".tiff"):
        shutil.move(filename, os.path.join(CYAN_DIR, filename))
        print(f"Moved TIFF: {filename}")

    elif filename.lower().endswith(".csv"):
        csv_files.append(filename)
        print(f"CSV uploaded: {filename}")

print("\nUpload complete.")
print("\nCyAN files:")
for filename in os.listdir(CYAN_DIR):
    print(" ", filename)

print("\nCSV files:")
for filename in csv_files:
    print(" ", filename)

In [ ]:
import os

csv_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".csv")
]

print("CSV files found:")
for f in csv_files:
    print(f)

In [ ]:
!ls /content/cyan_files

In [ ]:
PREVIOUS_CSV = "/content/algae_prototype_dataset_30k.csv"

In [ ]:
import os
import pandas as pd

print("=== CYAN FILES ===")

tif_files = sorted([
    f for f in os.listdir("/content/cyan_files")
    if f.lower().endswith((".tif", ".tiff"))
])

for f in tif_files:
    print(f)

print("\nNumber of TIFF files:", len(tif_files))

print("\n=== EXISTING CSV ===")

print("CSV:", PREVIOUS_CSV)

df = pd.read_csv(PREVIOUS_CSV)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nFirst 5 rows:")
display(df.head())

## 1. Config

In [ ]:
GEE_PROJECT_ID   = 'YOUR_GEE_PROJECT_ID'   # <-- your GEE project id
PREVIOUS_CSV     = '/content/algae_prototype_dataset_30k.csv'   # your existing 2,154-row real file (unchanged)
CYAN_DIR         = '/content/cyan_files'    # folder with CyAN .tif files for the NEW dates only

# New bloom dates -- sampled from the western-basin hotspot region
NEW_BLOOM_DATES = ['2022-08-25', '2022-09-05', '2022-09-10', '2023-08-15', '2023-08-25']

# New calm dates -- sampled from the full lake, same method as your original pre-bloom dates
NEW_CALM_DATES = ['2022-05-15', '2023-05-15']

COMPOSITE_WINDOW_DAYS = 4
CLOUD_PCT_MAX         = 40
NUM_SAMPLE_POINTS_BLOOM    = 80000   # per new bloom date -- hotspot region makes this affordable
NUM_SAMPLE_POINTS_PREBLOOM = 50000   # per new calm date
RANDOM_SEED           = 42

TARGET_PER_CLASS = 5000    # -> 10,000 total if both classes reach this
MIN_COVERAGE_FRACTION = 0.05

OUTPUT_CSV_PATH = '/content/algae_prototype_dataset_10k.csv'

## 2. Install, authenticate, mount Drive

In [ ]:
!pip install -U earthengine-api rasterio pyproj -q

import ee, os, re, time, glob
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import rasterio
import pyproj

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)

from google.colab import drive
drive.mount('/content/drive')
print('Earth Engine + Drive ready.')

## 3. Two regions: full lake vs. western-basin hotspot

In [ ]:
FULL_LAKE_ERIE = ee.Geometry.Rectangle([-83.5, 41.3, -78.8, 42.9])
WESTERN_BASIN_HOTSPOT = ee.Geometry.Rectangle([-83.5, 41.3, -82.5, 41.9])

def region_for_date(date_str):
    return WESTERN_BASIN_HOTSPOT if date_str in NEW_BLOOM_DATES else FULL_LAKE_ERIE

def sample_points_for_date(date_str):
    return NUM_SAMPLE_POINTS_BLOOM if date_str in NEW_BLOOM_DATES else NUM_SAMPLE_POINTS_PREBLOOM

print('New bloom dates (hotspot):', NEW_BLOOM_DATES)
print('New calm dates (full lake):', NEW_CALM_DATES)

## 4. Discover and date-parse the CyAN files for the new dates

In [ ]:
CYAN_FILENAME_RE = re.compile(r'L(\d{4})(\d{3})\.L3m_DAY_CYAN')

def parse_cyan_date(filepath):
    m = CYAN_FILENAME_RE.search(os.path.basename(filepath))
    if not m:
        return None
    year, doy = int(m.group(1)), int(m.group(2))
    return (datetime(year, 1, 1) + timedelta(days=doy - 1)).strftime('%Y-%m-%d')

dates_to_process = set(NEW_BLOOM_DATES) | set(NEW_CALM_DATES)

cyan_files = sorted(glob.glob(os.path.join(CYAN_DIR, '*.tif')))
assert cyan_files, f'No .tif files found in {CYAN_DIR}.'

date_file_pairs = []
for fp in cyan_files:
    d = parse_cyan_date(fp)
    if d is None:
        print('Could not parse date from filename, skipping:', fp)
        continue
    if d not in dates_to_process:
        print(f'File dated {d} is not in this run\'s new-date list, skipping:', os.path.basename(fp))
        continue
    date_file_pairs.append((d, fp))

found_dates = {d for d, _ in date_file_pairs}
missing = dates_to_process - found_dates
if missing:
    print('\nWARNING: no CyAN file found for:', missing, '-- download these first.')

print(f'\nProcessing {len(date_file_pairs)} new dates:')
for d, fp in sorted(date_file_pairs):
    region_name = 'HOTSPOT' if d in NEW_BLOOM_DATES else 'FULL LAKE'
    print(f'  {d}  ({region_name})  ->  {os.path.basename(fp)}')

## 5. Reusable feature-computation functions (same 31 features, region-parametrized)

In [ ]:
EPS = 1e-6

def mask_s2_clouds(image):
    scl = image.select('SCL')
    clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                        .And(scl.neq(10)).And(scl.neq(11)))
    return image.updateMask(clear)

def build_composite(center_date_str, window_days, region):
    center_date = datetime.strptime(center_date_str, '%Y-%m-%d')
    start = (center_date - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end   = (center_date + timedelta(days=window_days + 1)).strftime('%Y-%m-%d')
    coll = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_PCT_MAX))
            .map(mask_s2_clouds))
    n = coll.size().getInfo()
    if n == 0:
        return None, 0
    img = coll.select(['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']).median().divide(10000)
    return img.clip(region), n

def compute_feature_stack(img):
    B2, B3, B4  = img.select('B2'), img.select('B3'), img.select('B4')
    B5, B6, B7  = img.select('B5'), img.select('B6'), img.select('B7')
    B8, B8A     = img.select('B8'), img.select('B8A')
    B11, B12    = img.select('B11'), img.select('B12')

    NDWI  = img.normalizedDifference(['B3','B8']).rename('NDWI')
    NDVI  = img.normalizedDifference(['B8','B4']).rename('NDVI')
    MNDWI = img.normalizedDifference(['B3','B11']).rename('MNDWI')
    GNDVI = img.normalizedDifference(['B8','B3']).rename('GNDVI')
    NDCI  = img.normalizedDifference(['B5','B4']).rename('NDCI')

    FAI = img.expression(
        'B8 - (B4 + (B11 - B4) * ((842.0 - 665.0) / (1610.0 - 665.0)))',
        {'B8': B8, 'B4': B4, 'B11': B11}).rename('FAI')
    AFAI = img.expression(
        'B8A - (B4 + (B11 - B4) * ((865.0 - 665.0) / (1610.0 - 665.0)))',
        {'B8A': B8A, 'B4': B4, 'B11': B11}).rename('AFAI')
    SABI = B8.subtract(B4).divide(B2.add(B3).max(EPS)).rename('SABI')
    CI_GREEN = img.expression(
        'B3 - (B4 + (B5 - B4) * ((560.0 - 665.0) / (705.0 - 665.0)))',
        {'B3': B3, 'B4': B4, 'B5': B5}).rename('CI_GREEN')
    CI_RED_EDGE = img.expression(
        'B5 - (B4 + (B6 - B4) * ((705.0 - 665.0) / (740.0 - 665.0)))',
        {'B5': B5, 'B4': B4, 'B6': B6}).rename('CI_RED_EDGE')
    NDTI = img.normalizedDifference(['B4','B3']).rename('NDTI')
    NDMI = img.normalizedDifference(['B8','B11']).rename('NDMI')
    WATER_RATIO = B3.divide(B8.max(EPS)).rename('WATER_RATIO')
    TURBIDITY   = B4.divide(B3.max(EPS)).rename('TURBIDITY')
    FVI         = B8.divide(B4.max(EPS)).rename('FVI')

    index_stack = ee.Image.cat([
        NDWI, NDVI, MNDWI, GNDVI, NDCI, FAI, AFAI, SABI,
        CI_GREEN, CI_RED_EDGE, NDTI, NDMI, WATER_RATIO, TURBIDITY, FVI
    ])

    full_bands = img.addBands(index_stack)
    kernel = ee.Kernel.square(radius=1, units='pixels')
    ndci = index_stack.select('NDCI')
    NDCI_mean  = ndci.reduceNeighborhood(ee.Reducer.mean(), kernel).rename('NDCI_mean')
    NDCI_var   = ndci.reduceNeighborhood(ee.Reducer.variance(), kernel).rename('NDCI_var')
    NDCI_std   = ndci.reduceNeighborhood(ee.Reducer.stdDev(), kernel).rename('NDCI_std')
    NDCI_min   = ndci.reduceNeighborhood(ee.Reducer.min(), kernel).rename('NDCI_min')
    NDCI_max   = ndci.reduceNeighborhood(ee.Reducer.max(), kernel).rename('NDCI_max')
    NDCI_range = NDCI_max.subtract(NDCI_min).rename('NDCI_range')
    spatial_stack = ee.Image.cat([NDCI_mean, NDCI_var, NDCI_std, NDCI_min, NDCI_max, NDCI_range])

    feature_stack = full_bands.addBands(spatial_stack)
    assert len(feature_stack.bandNames().getInfo()) == 31
    return feature_stack, MNDWI, NDWI

print('Feature functions ready.')

## 6. Launch one export task per new date (concurrent, region-aware)

In [ ]:
def masked_pixel_count(mask_image, region, scale=200):
    try:
        return mask_image.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=region, scale=scale,
            maxPixels=1e9, bestEffort=True
        ).values().get(0).getInfo()
    except ee.EEException:
        return None

pending = []

for d, cyan_path in sorted(date_file_pairs):
    region = region_for_date(d)
    n_points = sample_points_for_date(d)
    region_name = 'HOTSPOT' if d in NEW_BLOOM_DATES else 'FULL LAKE'
    print(f'--- {d} ({region_name}, {n_points} pts) ---')

    img, n_scenes = build_composite(d, COMPOSITE_WINDOW_DAYS, region)
    if img is None:
        print('  No S2 scenes in window, skipping.')
        continue

    coverage = img.select('B2').mask().reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=200, maxPixels=1e9, bestEffort=True
    ).get('B2').getInfo()
    print(f'  Scenes: {n_scenes}  Coverage: {coverage:.3f}' if coverage is not None else '  Coverage check failed')
    if not coverage or coverage < MIN_COVERAGE_FRACTION:
        print('  Coverage too low, skipping.')
        continue

    feature_stack, MNDWI, NDWI = compute_feature_stack(img)

    water_mask = MNDWI.gt(0)
    wcount = masked_pixel_count(water_mask.selfMask().mask(), region)
    if wcount is not None and wcount < 500:
        water_mask = NDWI.gt(0)
        wcount = masked_pixel_count(water_mask.selfMask().mask(), region)
    print('  Water pixels (approx):', wcount)

    sampling_stack = (feature_stack
                       .updateMask(water_mask)
                       .addBands(ee.Image.pixelLonLat()))

    samples = sampling_stack.sample(
        region=region, scale=10, numPixels=n_points,
        seed=RANDOM_SEED, geometries=False, tileScale=8, dropNulls=True
    )

    export_name = f'algae_samples_new_{d.replace("-", "")}'
    task = ee.batch.Export.table.toDrive(
        collection=samples, description=export_name, fileNamePrefix=export_name,
        folder='GEE_Algae_Project', fileFormat='CSV'
    )
    task.start()
    pending.append({'date': d, 'cyan_path': cyan_path, 'task': task, 'export_name': export_name})
    print('  Export started:', export_name)

print(f'\n{len(pending)} export tasks launched.')

## 7. Wait for all exports to finish

In [ ]:
start = time.time()
while any(p['task'].active() for p in pending):
    states = [p['task'].status()['state'] for p in pending]
    done = sum(s in ('COMPLETED', 'FAILED', 'CANCELLED') for s in states)
    elapsed = int(time.time() - start)
    print(f'{done}/{len(pending)} finished ({elapsed}s elapsed)...')
    time.sleep(30)

for p in pending:
    p['final_state'] = p['task'].status()['state']

failed = [p for p in pending if p['final_state'] != 'COMPLETED']
succeeded = [p for p in pending if p['final_state'] == 'COMPLETED']
print(f'\nSucceeded: {len(succeeded)}   Failed/other: {len(failed)}')
for p in failed:
    print('  FAILED:', p['export_name'], '->', p['task'].status())

In [ ]:
CYAN_DIR      = '/content/cyan_files'
PREVIOUS_CSV  = '/content/algae_prototype_dataset_30k.csv'
OUTPUT_CSV_PATH = '/content/algae_prototype_dataset_10k.csv'
TARGET_PER_CLASS = 5000
RANDOM_SEED = 42

In [ ]:
import os, re, glob
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import rasterio
import pyproj

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
export_dir = '/content/drive/MyDrive/GEE_Algae_Project'
csv_files = sorted(glob.glob(os.path.join(export_dir, 'algae_samples_new_*.csv')))
print(f'Found {len(csv_files)} completed export CSVs in Drive:')
for f in csv_files:
    print(' ', os.path.basename(f))

EXPORT_NAME_RE = re.compile(r'algae_samples_new_(\d{8})\.csv')

def parse_export_date(filepath):
    m = EXPORT_NAME_RE.search(os.path.basename(filepath))
    if not m:
        return None
    s = m.group(1)
    return f'{s[0:4]}-{s[4:6]}-{s[6:8]}'

completed_dates = {}
for f in csv_files:
    d = parse_export_date(f)
    if d:
        completed_dates[d] = f

print('\nDates recovered:', sorted(completed_dates.keys()))

In [ ]:
CYAN_FILENAME_RE = re.compile(r'L(\d{4})(\d{3})\.L3m_DAY_CYAN')

def parse_cyan_date(filepath):
    m = CYAN_FILENAME_RE.search(os.path.basename(filepath))
    if not m:
        return None
    year, doy = int(m.group(1)), int(m.group(2))
    return (datetime(year, 1, 1) + timedelta(days=doy - 1)).strftime('%Y-%m-%d')

cyan_files = sorted(glob.glob(os.path.join(CYAN_DIR, '*.tif')))
assert cyan_files, f'No .tif files in {CYAN_DIR} -- re-upload your CyAN files first.'

cyan_by_date = {}
for fp in cyan_files:
    d = parse_cyan_date(fp)
    if d:
        cyan_by_date[d] = fp

succeeded = []
for d, csv_path in completed_dates.items():
    if d not in cyan_by_date:
        print(f'WARNING: no local CyAN file for {d} -- re-upload it, then re-run this cell.')
        continue
    succeeded.append({'date': d, 'cyan_path': cyan_by_date[d], 'export_name': f'algae_samples_new_{d.replace("-", "")}'})

print(f'\nReady to process {len(succeeded)} dates:')
for s in succeeded:
    print(' ', s['date'])

In [ ]:
def label_from_cyan(df_points, cyan_path):
    with rasterio.open(cyan_path) as src:
        cyan_arr = src.read(1)
        cyan_crs = src.crs
        cyan_transform = src.transform

    transformer = pyproj.Transformer.from_crs('EPSG:4326', cyan_crs, always_xy=True)
    xs, ys = transformer.transform(df_points['longitude'].values, df_points['latitude'].values)
    rows, cols = rasterio.transform.rowcol(cyan_transform, xs, ys)
    rows, cols = np.asarray(rows), np.asarray(cols)

    in_bounds = ((rows >= 0) & (rows < cyan_arr.shape[0]) &
                 (cols >= 0) & (cols < cyan_arr.shape[1]))

    dn = np.full(len(df_points), 255, dtype=np.int16)
    dn[in_bounds] = cyan_arr[rows[in_bounds], cols[in_bounds]]

    label = np.full(len(df_points), -1, dtype=int)
    label[dn == 0] = 0
    label[(dn >= 132) & (dn <= 253)] = 1

    df_points = df_points.copy()
    df_points['cyan_dn'] = dn
    df_points['label'] = label
    return df_points[df_points['label'] != -1].copy()


new_dfs = []
for p in succeeded:
    df_raw = pd.read_csv(completed_dates[p['date']])
    drop_cols = [c for c in ['system:index', '.geo'] if c in df_raw.columns]
    df_raw = df_raw.drop(columns=drop_cols)
    if len(df_raw) == 0:
        print(f"  {p['date']}: 0 sampled points, skipping.")
        continue

    df_labeled = label_from_cyan(df_raw, p['cyan_path'])
    df_labeled['source_date'] = p['date']
    new_dfs.append(df_labeled)

    n0 = (df_labeled['label'] == 0).sum()
    n1 = (df_labeled['label'] == 1).sum()
    print(f"  {p['date']}: {len(df_raw)} sampled -> {len(df_labeled)} labeled  "
          f"(no-bloom={n0}, bloom={n1})")

assert new_dfs, 'No date produced labeled rows -- check prints above.'
new_data = pd.concat(new_dfs, ignore_index=True)
print('\nNew data (pre-dedup):', new_data.shape)
print(new_data['label'].value_counts())

In [ ]:
previous_df = pd.read_csv(PREVIOUS_CSV)
print('Existing real data:', previous_df.shape)

feature_columns = [c for c in new_data.columns
                    if c not in ('longitude', 'latitude', 'cyan_dn', 'label', 'source_date')]
assert len(feature_columns) == 31

master = pd.concat([previous_df[new_data.columns], new_data], ignore_index=True)
before = len(master)
master = master.drop_duplicates(subset=feature_columns).reset_index(drop=True)
print(f'Combined: {master.shape}  (removed {before - len(master)} duplicates)')
print(master['label'].value_counts())
print(master.groupby(['source_date', 'label']).size())

achievable_per_class = min(master['label'].value_counts().min(), TARGET_PER_CLASS)
print(f'\nTarget: {TARGET_PER_CLASS}   Achievable: {achievable_per_class}')

balanced = (master.groupby('label', group_keys=False)
                   .apply(lambda g: g.sample(n=achievable_per_class, random_state=RANDOM_SEED))
                   .reset_index(drop=True))

n_nan = balanced[feature_columns].isna().sum().sum()
n_inf = np.isinf(balanced[feature_columns].to_numpy()).sum()
assert n_nan == 0 and n_inf == 0

balanced.to_csv(OUTPUT_CSV_PATH, index=False)
print('\nSaved:', OUTPUT_CSV_PATH)
print('Final shape:', balanced.shape)
print(balanced['label'].value_counts())

In [ ]:
from google.colab import files
files.download('/content/algae_prototype_dataset_10k.csv')

## 8. Per-date label lookup for the new dates

In [ ]:
def label_from_cyan(df_points, cyan_path):
    with rasterio.open(cyan_path) as src:
        cyan_arr = src.read(1)
        cyan_crs = src.crs
        cyan_transform = src.transform

    transformer = pyproj.Transformer.from_crs('EPSG:4326', cyan_crs, always_xy=True)
    xs, ys = transformer.transform(df_points['longitude'].values, df_points['latitude'].values)
    rows, cols = rasterio.transform.rowcol(cyan_transform, xs, ys)
    rows, cols = np.asarray(rows), np.asarray(cols)

    in_bounds = ((rows >= 0) & (rows < cyan_arr.shape[0]) &
                 (cols >= 0) & (cols < cyan_arr.shape[1]))

    dn = np.full(len(df_points), 255, dtype=np.int16)
    dn[in_bounds] = cyan_arr[rows[in_bounds], cols[in_bounds]]

    label = np.full(len(df_points), -1, dtype=int)
    label[dn == 0] = 0
    label[(dn >= 132) & (dn <= 253)] = 1

    df_points = df_points.copy()
    df_points['cyan_dn'] = dn
    df_points['label'] = label
    return df_points[df_points['label'] != -1].copy()


new_dfs = []
for p in succeeded:
    csv_path = f"/content/drive/MyDrive/GEE_Algae_Project/{p['export_name']}.csv"
    if not os.path.exists(csv_path):
        print('  Missing (Drive sync delay?):', csv_path)
        continue

    df_raw = pd.read_csv(csv_path)
    drop_cols = [c for c in ['system:index', '.geo'] if c in df_raw.columns]
    df_raw = df_raw.drop(columns=drop_cols)
    if len(df_raw) == 0:
        print(f"  {p['date']}: 0 sampled points, skipping.")
        continue

    df_labeled = label_from_cyan(df_raw, p['cyan_path'])
    df_labeled['source_date'] = p['date']
    new_dfs.append(df_labeled)

    n0 = (df_labeled['label'] == 0).sum()
    n1 = (df_labeled['label'] == 1).sum()
    print(f"  {p['date']}: {len(df_raw)} sampled -> {len(df_labeled)} labeled  "
          f"(no-bloom={n0}, bloom={n1})")

assert new_dfs, 'No new date produced any labeled rows -- check the per-date prints above.'
new_data = pd.concat(new_dfs, ignore_index=True)
print('\nNew data from today (pre-dedup):', new_data.shape)
print(new_data['label'].value_counts())

## 9. Combine with ALL of your existing real data (nothing dropped), deduplicate, balance

In [ ]:
previous_df = pd.read_csv(PREVIOUS_CSV)
print('Existing real data (untouched):', previous_df.shape)
print(previous_df['label'].value_counts())

feature_columns = [c for c in new_data.columns
                    if c not in ('longitude', 'latitude', 'cyan_dn', 'label', 'source_date')]
assert len(feature_columns) == 31

master = pd.concat([previous_df[new_data.columns], new_data], ignore_index=True)
print('\nCombined (pre-dedup):', master.shape)

before = len(master)
master = master.drop_duplicates(subset=feature_columns).reset_index(drop=True)
print(f'Removed {before - len(master)} duplicate rows.')

print('\nAvailable per class:', master['label'].value_counts().to_dict())
print('Per-date breakdown:')
print(master.groupby(['source_date', 'label']).size())

achievable_per_class = min(master['label'].value_counts().min(), TARGET_PER_CLASS)
print(f'\nTarget per class: {TARGET_PER_CLASS}   Achievable: {achievable_per_class}')
if achievable_per_class < TARGET_PER_CLASS:
    print('WARNING: still short -- see the note at the end of this notebook.')

balanced = (master.groupby('label', group_keys=False)
                   .apply(lambda g: g.sample(n=achievable_per_class, random_state=RANDOM_SEED))
                   .reset_index(drop=True))

print('\nFinal balanced shape:', balanced.shape)
print(balanced['label'].value_counts())
print('Dates represented:', sorted(balanced['source_date'].unique()))

## 10. Final sanity checks and save

In [ ]:
X = balanced[feature_columns]
n_nan = X.isna().sum().sum()
n_inf = np.isinf(X.to_numpy()).sum()
n_dup = balanced.duplicated(subset=feature_columns).sum()
print('NaN:', n_nan, ' Inf:', n_inf, ' Remaining duplicates:', n_dup)
assert n_nan == 0 and n_inf == 0 and n_dup == 0

balanced.to_csv(OUTPUT_CSV_PATH, index=False)
print('\nSaved:', OUTPUT_CSV_PATH)
print('Final shape:', balanced.shape)

## If still short of 5,000/class
Add more dates from the same high-severity years (2022, 2023) rather than going back to weaker
2024 dates -- check NOAA's Lake Erie HAB Seasonal Retrospective bulletins for the exact peak
window each year, since a couple of days off-peak can matter. Re-run only Sections 6-10 with the
new dates added to `NEW_BLOOM_DATES`/`NEW_CALM_DATES` -- nothing else needs to change.
